In [0]:
storage_account = "dmpro2"
access_key = ""   # hard-coded

spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", access_key)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *


catalog_name = "dm_pro2"
base_path = "abfss://rawdata@dmpro2.dfs.core.windows.net/data"


# Define schemas
orders_schema = StructType([
    StructField("OrderID", StringType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("ProductID", StringType(), True),
    StructField("OrderDate", StringType(), True),
    StructField("Quantity", StringType(), True),
    StructField("TotalAmount", StringType(), True),
    StructField("Status", StringType(), True),
])


returns_schema = StructType([
    StructField("ReturnID", StringType(), True),
    StructField("OrderID", StringType(), True),
    StructField("ReturnDate", StringType(), True),
    StructField("Reason", StringType(), True),
    StructField("RefundAmount", StringType(), True),
])


# Auto Loader ingestion function
def autoload_to_bronze(entity: str, schema: StructType, path: str):
    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(schema)
        .load(path)
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("ingested_at", current_timestamp())
    )
    
    (df.writeStream
        .format("delta")
        .option("checkpointLocation", f"{base_path}/checkpoints/{entity}")
        .outputMode("append")
        .trigger(availableNow=True).table(f"{catalog_name}.bronze.{entity}"))


# Ingest Orders and Returns incrementally
autoload_to_bronze("orders", orders_schema, f"{base_path}/orders/")
autoload_to_bronze("returns", returns_schema, f"{base_path}/returns/")
